# CNN Training for Facial Emotion Recognition
Train a CNN on a FER2013-style directory dataset and save the model artifacts.

In [ ]:
from pathlib import Path
import pickle

import cv2
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import tensorflow as tf
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Dropout, Flatten, Dense, Input
from tensorflow.keras.models import Sequential
from tensorflow.keras.utils import to_categorical

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

DATASET_DIR = Path("../images")
MODEL_OUTPUT = Path("../models/expressiondetector_modern.keras")
ENCODER_OUTPUT = Path("../models/label_encoder.pkl")
IMAGE_SIZE = (48, 48)
CLASS_NAMES = ["angry", "disgust", "fear", "happy", "neutral", "sad", "surprise"]

In [ ]:
def load_split(split_dir: Path):
    images = []
    labels = []

    for class_dir in sorted(split_dir.iterdir()):
        if not class_dir.is_dir():
            continue

        for image_path in sorted(class_dir.glob("*")):
            image = cv2.imread(str(image_path), cv2.IMREAD_GRAYSCALE)
            if image is None:
                continue

            image = cv2.resize(image, IMAGE_SIZE)
            image = image.astype(np.float32) / 255.0
            images.append(image[..., np.newaxis])
            labels.append(class_dir.name)

    return np.array(images, dtype=np.float32), np.array(labels)

train_dir = DATASET_DIR / "train"
test_dir = DATASET_DIR / "test"

x_train_full, y_train_full = load_split(train_dir)
x_test, y_test_raw = load_split(test_dir)

print(f"Loaded training samples: {len(x_train_full)}")
print(f"Loaded test samples: {len(x_test)}")

In [ ]:
label_encoder = LabelEncoder()
label_encoder.fit(CLASS_NAMES)

y_train_encoded = label_encoder.transform(y_train_full)
y_test_encoded = label_encoder.transform(y_test_raw)

y_train = to_categorical(y_train_encoded, num_classes=7)
y_test = to_categorical(y_test_encoded, num_classes=7)

x_train, x_val, y_train, y_val = train_test_split(
    x_train_full,
    y_train,
    test_size=0.1,
    random_state=SEED,
    stratify=y_train_encoded
)

print(x_train.shape, x_val.shape, x_test.shape)

In [ ]:
model = Sequential([
    Input(shape=(48, 48, 1)),
    Conv2D(128, (3, 3), activation="relu", padding="same"),
    MaxPooling2D(pool_size=(2, 2)),
    Dropout(0.4),
    Conv2D(256, (3, 3), activation="relu", padding="same"),
    MaxPooling2D(pool_size=(2, 2)),
    Dropout(0.4),
    Conv2D(512, (3, 3), activation="relu", padding="same"),
    MaxPooling2D(pool_size=(2, 2)),
    Dropout(0.4),
    Conv2D(512, (3, 3), activation="relu", padding="same"),
    MaxPooling2D(pool_size=(2, 2)),
    Dropout(0.4),
    Flatten(),
    Dense(512, activation="relu"),
    Dropout(0.4),
    Dense(256, activation="relu"),
    Dropout(0.3),
    Dense(7, activation="softmax")
])

model.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])
model.summary()

In [ ]:
callbacks = [
    tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3, verbose=1),
    tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True, verbose=1)
]

history = model.fit(
    x_train,
    y_train,
    validation_data=(x_val, y_val),
    epochs=25,
    batch_size=128,
    callbacks=callbacks
)

test_loss, test_accuracy = model.evaluate(x_test, y_test, verbose=0)
print({"test_loss": test_loss, "test_accuracy": test_accuracy})

In [ ]:
MODEL_OUTPUT.parent.mkdir(parents=True, exist_ok=True)
model.save(MODEL_OUTPUT)

with ENCODER_OUTPUT.open("wb") as handle:
    pickle.dump(label_encoder, handle)

print(f"Saved model to {MODEL_OUTPUT}")
print(f"Saved label encoder to {ENCODER_OUTPUT}")